In [ ]:
# Install mlflow only when running in Google Colab (it's pre-installed in the sandbox).
# COLAB_RELEASE_TAG is an environment variable set automatically by Colab at runtime;
# its presence is the standard way to detect the Colab environment.
!pip install mlflow --quiet

import os

# Choose the MLflow tracking server based on the runtime environment.
# - In Colab: write experiment data to a local temp directory (no server needed).
# - In the sandbox: use the shared MLflow container accessible at http://mlflow:5000
#   (the hostname "mlflow" resolves via Docker's internal DNS on the sandbox network).
MLFLOW_URI = "file:///tmp/mlruns" if "COLAB_RELEASE_TAG" in os.environ else "http://mlflow:5000"


# ML Basics: Training Your First Model

This notebook mirrors the **ML Basics** guide at [learnmlops.ops4life.com/guides/ml-basics/](https://learnmlops.ops4life.com/guides/ml-basics/).

Train a Random Forest classifier on the Iris dataset, log the experiment with MLflow, and inspect the results in the tracking UI. This is the starting point of the MLOps journey — before pipelines, before production.

**What we'll cover:**
1. Load the Iris dataset
2. Split into train and test sets
3. Train with MLflow experiment tracking
4. Inspect results

In [ ]:
from sklearn.datasets import load_iris          # Built-in dataset: 150 samples, 4 numeric features, 3 classes
from sklearn.model_selection import train_test_split  # Splits data into train/test while preserving class ratios
from sklearn.ensemble import RandomForestClassifier   # Ensemble of decision trees; robust baseline for classification
from sklearn.metrics import accuracy_score, classification_report  # Evaluation helpers
import mlflow              # Experiment tracking: log params, metrics, and model artifacts
import mlflow.sklearn      # sklearn integration: serialise and log sklearn models automatically

# Tell MLflow where to write run data.
# set_tracking_uri() must be called before any run starts.
mlflow.set_tracking_uri(MLFLOW_URI)

# Group all runs for this task under one named experiment.
# If the experiment doesn't exist yet, MLflow creates it automatically.
# Experiments keep runs for different tasks separate in the UI.
mlflow.set_experiment('iris-classification')


## Step 1: Load the Iris dataset

In [ ]:
iris = load_iris()

# iris.data   → numpy array of shape (150, 4): the feature matrix (sepal/petal length/width)
# iris.target → numpy array of shape (150,): integer class labels 0, 1, or 2
X, y = iris.data, iris.target

# Human-readable names for features and classes — used for reporting and labelling axes.
# iris.feature_names: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
# iris.target_names:  ['setosa', 'versicolor', 'virginica']
feature_names = iris.feature_names
class_names   = iris.target_names

print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features')
print(f'Classes: {list(class_names)}')
print(f'Features: {list(feature_names)}')


## Step 2: Split into train and test sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,    # Hold out 20% (30 samples) for evaluation. Common choice: 10–30%.
    random_state=42,  # Fix the random seed so the split is identical on every run.
                      # Without this, each run gets a different split and metrics are
                      # not comparable between runs.
    stratify=y,       # Maintain class proportions in both splits.
                      # Without stratify, a 150-sample dataset could have all 50 setosa
                      # end up in train, making the test set unreliable for that class.
)

print(f'Train: {X_train.shape[0]} samples')
print(f'Test:  {X_test.shape[0]} samples')


## Step 3: Train with MLflow tracking

In [ ]:
# mlflow.start_run() opens a new run and returns a context object.
# Using it as a context manager (with ...) automatically ends the run when the
# block exits — even if an exception is raised. This prevents orphaned "RUNNING" runs.
with mlflow.start_run(run_name='rf-baseline'):
    n_estimators = 100   # Number of decision trees in the forest.
    max_depth    = 5     # Max tree depth — controls overfitting. None = grow until pure.

    # log_param() records hyperparameters — values set BEFORE training.
    # These appear in the "Parameters" tab in the MLflow UI and are used to
    # reproduce or compare runs. Log anything that changes model behaviour.
    mlflow.log_param('n_estimators', n_estimators)
    mlflow.log_param('max_depth', max_depth)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42,   # Fix internal random state for reproducible tree building
    )
    # fit() trains on TRAINING data only. The test set must not influence training.
    model.fit(X_train, y_train)

    # Predict labels for the test set (held-out data the model has never seen).
    y_pred = model.predict(X_test)

    # Accuracy = fraction of correct predictions. Simple but misleading on imbalanced data.
    acc = accuracy_score(y_test, y_pred)

    # log_metric() records evaluation results — values computed AFTER training.
    # Metrics appear in the "Metrics" tab and are used to compare runs in the UI.
    mlflow.log_metric('accuracy', acc)

    # log_model() serialises the trained sklearn model (via joblib) and stores it
    # as an artifact in the run. The second argument ('model') is the artifact path
    # inside the run's artifact directory.
    # This artifact can later be loaded with mlflow.sklearn.load_model(model_uri).
    mlflow.sklearn.log_model(model, 'model')

    # run_id uniquely identifies this run — use it to reload the model later:
    # mlflow.sklearn.load_model(f"runs:/{run_id}/model")
    run_id = mlflow.active_run().info.run_id
    print(f'Run ID: {run_id}')
    print(f'Accuracy: {acc:.4f}')


MLflow recorded the run parameters (`n_estimators`, `max_depth`), the accuracy metric, and the serialised model artifact. Open the MLflow UI at [mlflow.learnmlops.ops4life.com](https://mlflow.learnmlops.ops4life.com) and navigate to the **iris-classification** experiment to see the run.

## Step 4: Inspect results

In [ ]:
# classification_report prints per-class precision, recall, F1, and support.
# - Precision: of all samples predicted as class X, how many actually are X?
# - Recall:    of all samples that ARE class X, how many did we correctly predict?
# - F1:        harmonic mean of precision and recall — good single metric for imbalanced classes
# - Support:   number of true samples for each class in the test set
# target_names maps integer labels (0, 1, 2) to human-readable class names in the output.
print(classification_report(y_test, y_pred, target_names=class_names))


## Next Steps

- Explore the MLflow UI: compare runs, view logged artifacts, and register a model
- Continue to the pipeline: `02-pipeline/dataset-pipeline/ingest.ipynb`
- Full guide: [learnmlops.ops4life.com/guides/ml-basics/](https://learnmlops.ops4life.com/guides/ml-basics/)